In [141]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
plt.style.use('classic')
import numpy as np
import calendar
import os
import locale
locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
sns.set()

from IPython.display import HTML


In [142]:
print('the current working directory is:', os.getcwd())

the current working directory is: d:\GitHub\Sala_Situacion_Dinagua\Status_Outlook_Bulletin\notebook


In [143]:
def obtener_segundo_mes(row):
    ano = int(row['year'])
    mes = int(row['month'])
    _, num_dias = calendar.monthrange(ano, mes)
    segundos_en_dia = 24 * 60 * 60
    return num_dias * segundos_en_dia

def cargar_y_procesar(ruta, agregar_dias=False):
    # Cargar y renombrar columnas
    df = pd.read_csv(ruta).rename(columns={'-1': 'year', '-1.1': 'month'}).copy()
    
    # Construir el índice temporal directamente
    fechas = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1))
    df = df.set_index(fechas)
    df.index.name = 'date'
    
    if agregar_dias:
        # .assign() evita la fragmentación al crear columnas
        df = df.assign(days_in_month=df.index.days_in_month)
        
    return df

In [144]:
# Carga de datos
ESCORRENTIA_total = cargar_y_procesar('../waterbalance/balance_hidrico_regional/output_modelo/Escorrentia_total.csv', agregar_dias=True)
ESCORRENTIA_sup   = cargar_y_procesar('../waterbalance/balance_hidrico_regional/output_modelo/Escorrentia_sup.csv', agregar_dias=True)
ESCORRENTIA_sub   = cargar_y_procesar('../waterbalance/balance_hidrico_regional/output_modelo/Escorrentia_sub.csv', agregar_dias=True)

PRECIP = cargar_y_procesar('../waterbalance/balance_hidrico_regional/output_modelo/Pmedias.csv')
ETR    = cargar_y_procesar('../waterbalance/balance_hidrico_regional/output_modelo/ETR.csv')
SM     = cargar_y_procesar('../waterbalance/balance_hidrico_regional/output_modelo/HumedadSuelo.csv')

In [145]:
# Convert 'Year' and 'Month' columns to integers
ESCORRENTIA_total['year'] = ESCORRENTIA_total['year'].astype(int)
ESCORRENTIA_total['month'] = ESCORRENTIA_total['month'].astype(int)

In [146]:
ESCORRENTIA_total['seconds_in_month'] = ESCORRENTIA_total.apply(obtener_segundo_mes, axis=1)

In [147]:
# Importar cuencas niveles 2 y 3
CUENCA_NIVEL_3 = pd.read_csv(f'../waterbalance/balance_hidrico_regional/output_modelo/cuenca_nivel3.csv',index_col="Codigo")
CUENCA_NIVEL_2 = pd.read_csv(f'../waterbalance/balance_hidrico_regional/output_modelo/cuenca_nivel2.csv',index_col="Codigo")

In [148]:
# Convertir escorrentia a caudales
ESCORRENTIA_SELECTION = ESCORRENTIA_total.drop(['year','month','days_in_month','seconds_in_month'],axis=1)
CAUDAL = pd.DataFrame(ESCORRENTIA_SELECTION.values*1000*CUENCA_NIVEL_3.values,columns=ESCORRENTIA_SELECTION.columns)
CAUDAL['days_in_month'] = ESCORRENTIA_total['days_in_month'].values
CAUDAL = CAUDAL.loc[:, CAUDAL.columns != 'days_in_month'].divide(CAUDAL["days_in_month"]*24*3600, axis="index")
# Definir columns de date y año
CAUDAL['date'] = ESCORRENTIA_total.index.values
CAUDAL = CAUDAL.set_index('date')
CAUDAL['year'] = ESCORRENTIA_total['year'].values
CAUDAL['month'] = ESCORRENTIA_total['month'].values

In [149]:
# Convertir caudal a volumen
VOLUME = pd.DataFrame(CAUDAL.drop(['year','month'],axis=1))
VOLUME['seconds_in_month'] = ESCORRENTIA_total['seconds_in_month'].values
VOLUME = VOLUME.loc[:, VOLUME.columns != 'seconds_in_month'].multiply(VOLUME["seconds_in_month"], axis="index")
VOLUME['date'] = CAUDAL.index.values
VOLUME = VOLUME.set_index('date')
VOLUME['year'] = CAUDAL['year'].values
VOLUME['month'] = CAUDAL['month'].values

In [150]:
# Seleccionar codcuenca n2
codcuenca_n2 = 61

# Buscar columnas que empiezan con el codigo de cuenca nivel 2
filter_col = [col for col in CAUDAL if col.startswith(str(codcuenca_n2))]

# Obtener variables en base a los codigos de cuenca nivel 3 que corresponden a las cuencas nivel 2
CAUDAL_SELECTED = CAUDAL[filter_col]
PRECIP_SELECTED = PRECIP[filter_col]
ETR_SELECTED = ETR[filter_col]
SM_SELECTED = SM[filter_col]
ESCORRENTIA_TOTAL_SELECTED = ESCORRENTIA_total[filter_col]
ESCORRENTIA_SUP_SELECTED = ESCORRENTIA_sup[filter_col]
ESCORRENTIA_SUB_SELECTED = ESCORRENTIA_sub[filter_col]
VOLUME_SELECTED = VOLUME[filter_col]

In [151]:
# Agregar variables

# Caudal
CAUDAL_N2 = pd.DataFrame()
CAUDAL_N2['year'] = CAUDAL['year']
CAUDAL_N2['month'] = CAUDAL['month']
CAUDAL_N2['discharge'] = CAUDAL_SELECTED.sum(axis=1)

CAUDAL_N2['date'] = pd.to_datetime(dict(year=CAUDAL_N2['year'],month=CAUDAL_N2['month'],day=1))
CAUDAL_N2 = CAUDAL_N2.set_index('date')

# Precip
PRECIP_N2 = pd.DataFrame()
PRECIP_N2['year'] = PRECIP['year']
PRECIP_N2['month'] = PRECIP['month']
PRECIP_N2['precip'] = PRECIP_SELECTED.mean(axis=1)

PRECIP_N2['date'] = pd.to_datetime(dict(year=PRECIP_N2['year'],month=PRECIP_N2['month'],day=1))
PRECIP_N2 = PRECIP_N2.set_index('date')

# Escorrentia Total
ESCORRENTIA_TOTAL_N2 = pd.DataFrame()
ESCORRENTIA_TOTAL_N2['year'] = ESCORRENTIA_total['year']
ESCORRENTIA_TOTAL_N2['month'] = ESCORRENTIA_total['month']
ESCORRENTIA_TOTAL_N2['runoff'] = ESCORRENTIA_TOTAL_SELECTED.mean(axis=1)

ESCORRENTIA_TOTAL_N2['date'] = pd.to_datetime(dict(year=ESCORRENTIA_TOTAL_N2['year'],month=ESCORRENTIA_TOTAL_N2['month'],day=1))
ESCORRENTIA_TOTAL_N2 = ESCORRENTIA_TOTAL_N2.set_index('date')

# Escorrentia Superficial
ESCORRENTIA_SUP_N2 = pd.DataFrame()
ESCORRENTIA_SUP_N2['year'] = ESCORRENTIA_sup['year']
ESCORRENTIA_SUP_N2['month'] = ESCORRENTIA_sup['month']
ESCORRENTIA_SUP_N2['runoff'] = ESCORRENTIA_SUP_SELECTED.mean(axis=1)

ESCORRENTIA_SUP_N2['date'] = pd.to_datetime(dict(year=ESCORRENTIA_SUP_N2['year'],month=ESCORRENTIA_SUP_N2['month'],day=1))
ESCORRENTIA_SUP_N2 = ESCORRENTIA_SUP_N2.set_index('date')

# Escorrentia Subterranea
ESCORRENTIA_SUB_N2 = pd.DataFrame()
ESCORRENTIA_SUB_N2['year'] = ESCORRENTIA_sub['year']
ESCORRENTIA_SUB_N2['month'] = ESCORRENTIA_sub['month']
ESCORRENTIA_SUB_N2['runoff'] = ESCORRENTIA_SUB_SELECTED.mean(axis=1)

ESCORRENTIA_SUB_N2['date'] = pd.to_datetime(dict(year=ESCORRENTIA_SUB_N2['year'],month=ESCORRENTIA_SUB_N2['month'],day=1))
ESCORRENTIA_SUB_N2 = ESCORRENTIA_SUB_N2.set_index('date')

# ETR
ETR_N2 = pd.DataFrame()
ETR_N2['year'] = ETR['year']
ETR_N2['month'] = ETR['month']
ETR_N2['etr'] = ETR_SELECTED.mean(axis=1)

ETR_N2['date'] = pd.to_datetime(dict(year=ETR_N2['year'],month=ETR_N2['month'],day=1))
ETR_N2 = ETR_N2.set_index('date')

# Humedad Suelo
SM_N2 = pd.DataFrame()
SM_N2['year'] = SM['year']
SM_N2['month'] = SM['month']
SM_N2['soilmoisture'] = SM_SELECTED.mean(axis=1)

SM_N2['date'] = pd.to_datetime(dict(year=SM_N2['year'],month=SM_N2['month'],day=1))
SM_N2 = SM_N2.set_index('date')

# Volumen (Aqui se convierte el volumen de m3 a hm3)
VOLUME_N2 = pd.DataFrame()
VOLUME_N2['year'] = VOLUME['year']
VOLUME_N2['month'] = VOLUME['month']
VOLUME_N2['volume'] = VOLUME_SELECTED.sum(axis=1) / 1e6

VOLUME_N2['date'] = pd.to_datetime(dict(year=VOLUME_N2['year'],month=VOLUME_N2['month'],day=1))
VOLUME_N2 = VOLUME_N2.set_index('date')

In [152]:
# Graficar los ultimos 12 meses
# Quitar comentario si desea graficar el caudal 
CAUDAL_N2_PLOT = CAUDAL_N2

# Graficar los ultimos valores 
VOLUME_N2_PLOT = VOLUME_N2.tail(12)
SM_N2_PLOT = SM_N2.tail(12)
PRECIP_N2_PLOT = PRECIP_N2.tail(12)
ESCORRENTIA_TOTAL_N2_PLOT = ESCORRENTIA_TOTAL_N2.tail(12)
ESCORRENTIA_SUP_N2_PLOT = ESCORRENTIA_SUP_N2.tail(12)
ESCORRENTIA_SUB_N2_PLOT = ESCORRENTIA_SUB_N2.tail(12)
ETR_N2_PLOT = ETR_N2.tail(12)

In [153]:
# merge all the selected dataframes for plotting and define name columns 
df4 = pd.merge(pd.merge(pd.merge(PRECIP_N2_PLOT,ETR_N2_PLOT,on=['date','year','month']),SM_N2_PLOT,on=['date','year','month']),ESCORRENTIA_TOTAL_N2_PLOT,on=['date','year','month'])
df4 = pd.merge(df4,ESCORRENTIA_SUP_N2_PLOT,on=['date','year','month'])
df4 = pd.merge(df4,ESCORRENTIA_SUB_N2_PLOT,on=['date','year','month'])
df4 = pd.merge(df4,VOLUME_N2_PLOT,on=['date','year','month'])

df4.columns = ['year','month','RAINFALL','ETR','SOIL MOISTURE','TOTAL RUNOFF','RUNOFF','GROUNDWATER','VOLUME']
df4 = df4.drop(['year', 'month','VOLUME','TOTAL RUNOFF'], axis=1)

In [154]:
df4

,RAINFALL,ETR,SOIL MOISTURE,RUNOFF,GROUNDWATER
date,,,,,
2025-11-01,60.307714,95.140286,0.000000,0.000000,5.178886
2025-12-01,39.634429,39.634429,0.000000,0.000000,1.924343
2026-01-01,67.326857,67.326857,0.000000,0.000000,0.715044
2026-02-01,66.951857,66.951857,0.000000,0.000000,0.265693
2026-03-01,119.414286,110.108571,7.751471,0.089075,0.670764
2026-04-01,124.617143,62.752857,63.199000,0.866849,2.764795
2026-05-01,55.054143,38.858143,74.281714,0.770311,4.043700
2026-06-01,12.094714,21.284714,65.091857,0.000000,2.536114
2026-07-01,88.036143,32.503000,101.261286,5.439952,6.378783


In [165]:
# 1. Etiquetas de fechas para el eje X (%b-%y en mayúsculas)
etiquetas_x = [d.strftime("%b-%y").upper() for d in df4.index]

fig = go.Figure()

# 2. Entrada (Positiva)
fig.add_trace(
    go.Bar(
        x=etiquetas_x,
        y=df4["RAINFALL"],
        name="Precipitación",
        marker_color="rgba(31, 119, 180, 0.85)",  # #1f77b4
        customdata=df4["RAINFALL"],
        hovertemplate="<b>%{x}</b><br>Precipitación: %{customdata:.2f} mm<extra></extra>",
    )
)

# 3. Salidas y Almacenamientos (Negativas apiladas)
fig.add_trace(
    go.Bar(
        x=etiquetas_x,
        y=-df4["ETR"],
        name="Evapotranspiración Real",
        marker_color="rgba(255, 127, 14, 0.85)",  # #ff7f0e
        customdata=df4["ETR"],
        hovertemplate="<b>%{x}</b><br>ETR: %{customdata:.2f} mm<extra></extra>",
    )
)

fig.add_trace(
    go.Bar(
        x=etiquetas_x,
        y=-df4["RUNOFF"],
        name="Escurrimiento Superficial",
        marker_color="rgba(44, 160, 44, 0.85)",  # #2ca02c
        customdata=df4["RUNOFF"],
        hovertemplate="<b>%{x}</b><br>RUNOFF: %{customdata:.2f} mm<extra></extra>",
    )
)

fig.add_trace(
    go.Bar(
        x=etiquetas_x,
        y=-df4["SOIL MOISTURE"],
        name="Humedad de Suelo",
        marker_color="rgba(140, 86, 75, 0.85)",  # #8c564b
        customdata=df4["SOIL MOISTURE"],
        hovertemplate="<b>%{x}</b><br>SOIL MOISTURE: %{customdata:.2f} mm<extra></extra>",
    )
)

fig.add_trace(
    go.Bar(
        x=etiquetas_x,
        y=-df4["GROUNDWATER"],
        name="Almacenamiento Subterráneo",
        marker_color="rgba(148, 103, 189, 0.85)",  # #9467bd
        customdata=df4["GROUNDWATER"],
        hovertemplate="<b>%{x}</b><br>GROUNDWATER: %{customdata:.2f} mm<extra></extra>",
    )
)

# 4. Línea divisoria horizontal en Y=0
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    line_width=1,
)

# 5. Configuración de diseño con tema y fondos por defecto
fig.update_layout(
    template="plotly",  # Tema estándar de Plotly
    title=dict(
        text="Balance Hídrico Mensual: Entradas, Salidas y Almacenamiento",
        x=0.5,
        xanchor="center",
        font=dict(size=17),
    ),
    barmode="relative",
    bargap=0.45,
    yaxis=dict(
        title=dict(
            text="<b>Lámina de Agua (mm / mes)</b>",
            font=dict(size=14),
        ),
        range=[-400, 400],
    ),
    xaxis=dict(
        type="category",
        tickangle=0,
    ),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
    ),
    width=1100,
    height=600,
    margin=dict(l=60, r=40, t=60, b=100),
)

fig.show()